<h2 style="
background-color:#F8D7DA;
color:black;
padding:10px;
border-radius:8px;
font-weight:bold;
font-style:italic;
">
Task 3: Customer Profile Building
</h>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("cleaned_jp_morgan.csv")

In [3]:
df.head()

,TransactionID,CustomerID,AccountID,AccountType,TransactionType,Product,Firm,Region,Manager,TransactionDate,TransactionAmount,AccountBalance,RiskScore,CreditRating,TenureMonths,Year,Month
0,3,CUST2412,ACC80131,loan,withdrawal,Personal Loan,Firm C,West,Manager 3,2023-08-06,33759.69057,126486.40830,0.225824,611,89,2023,8
1,32,CUST1467,ACC74631,current,withdrawal,Home Loan,Firm D,North,Manager 2,2023-11-08,69319.19933,24834.76291,0.335717,817,174,2023,11
2,9,CUST2699,ACC39482,loan,withdrawal,Credit Card,Firm D,West,Manager 4,2024-05-15,42831.48483,123007.43530,0.572453,332,31,2024,5
3,42,CUST9535,ACC82947,current,withdrawal,Home Loan,Firm A,South,Manager 4,2023-04-30,70903.79697,73073.64225,0.571993,626,92,2023,4
4,166,CUST7459,ACC39500,credit,payment,Home Loan,Firm D,South,Manager 4,2023-02-16,21948.97355,113405.32820,0.380675,411,13,2023,2


**<h3 style="color:blue;">3.1 Group Accounts by Activity Level.</h3>**

#### Rubric:
##### • High Activity   : Top 25% of accounts by transaction frequency
##### • Medium Activity : Middle 50% of accounts
##### • Low Activity    : Bottom 25% of accounts

In [9]:

activity_summary = (
    df.groupby("AccountID")["TransactionID"]
      .count()
      .reset_index(name="Transaction_Count")
)

low_limit = activity_summary["Transaction_Count"].quantile(0.25)
high_limit = activity_summary["Transaction_Count"].quantile(0.75)

def classify_activity(count):
    if count >= high_limit:
        return "High"
    elif count <= low_limit:
        return "Low"
    else:
        return "Medium"

activity_summary["Activity_Level"] = activity_summary["Transaction_Count"].apply(classify_activity)

activity_summary

,AccountID,Transaction_Count,Activity_Level
0,ACC10117,1,Low
1,ACC10996,4,Medium
2,ACC11062,4,Medium
3,ACC11188,3,Low
4,ACC11285,6,High
...,...,...,...
189,ACC97225,5,High
190,ACC97411,6,High
191,ACC99117,6,High
192,ACC99409,2,Low


**<h3 style="color:blue;">3.2 Segment Customers by Average Balance & Transaction Volume.</h3>**

#### Customer Segmentation

In [11]:

customer_profile = (
    df.groupby("AccountID")
      .agg(
          Average_Balance=("AccountBalance","mean"),
          Transaction_Volume=("TransactionAmount","sum")
      )
      .reset_index()
)

balance_q1 = customer_profile["Average_Balance"].quantile(0.25)
balance_q3 = customer_profile["Average_Balance"].quantile(0.75)

volume_q1 = customer_profile["Transaction_Volume"].quantile(0.25)
volume_q3 = customer_profile["Transaction_Volume"].quantile(0.75)

def balance_segment(x):
    if x >= balance_q3:
        return "High"
    elif x <= balance_q1:
        return "Low"
    return "Medium"

def volume_segment(x):
    if x >= volume_q3:
        return "High"
    elif x <= volume_q1:
        return "Low"
    return "Medium"

customer_profile["Balance_Segment"] = customer_profile["Average_Balance"].apply(balance_segment)

customer_profile["Volume_Segment"] = customer_profile["Transaction_Volume"].apply(volume_segment)

customer_profile

,AccountID,Average_Balance,Transaction_Volume,Balance_Segment,Volume_Segment
0,ACC10117,90780.256640,56317.920060,High,Low
1,ACC10996,64046.568590,223757.534516,Medium,Medium
2,ACC11062,62784.100737,265928.583340,Low,Medium
3,ACC11188,80558.926400,116557.954500,Medium,Low
4,ACC11285,95745.546255,373189.528410,High,High
...,...,...,...,...,...
189,ACC97225,80994.270410,150029.530282,Medium,Medium
190,ACC97411,61783.633875,310826.633750,Low,High
191,ACC99117,80478.450622,386544.981030,Medium,High
192,ACC99409,32962.941265,169048.801460,Low,Medium


**<h3 style="color:blue;">3.3 High Net Inflow Accounts.</h3>**

#### Assign Positive and Negative Values Based on Transaction Type

In [24]:


transaction_sign = {
    "deposit": 1,
    "withdrawal": -1,
    "payment": -1,
    "transfer": -1
}

df["Net_Amount"] = (
    df["TransactionAmount"] *
    df["TransactionType"].map(transaction_sign)
)

In [25]:
net_inflow = (
    df.groupby("AccountID")["Net_Amount"]
      .sum()
      .reset_index()
)

high_net_accounts = net_inflow.nlargest(10, "Net_Amount")

high_net_accounts

,AccountID,Net_Amount
109,ACC54589,225122.495180
131,ACC67713,189071.785288
141,ACC74631,150954.104630
169,ACC87602,150860.567520
26,ACC22036,145164.795582
1,ACC10996,124107.378684
115,ACC57872,118892.676040
72,ACC39500,118064.429594
42,ACC28292,93721.351460
100,ACC50817,83157.353560


**<h3 style="color:blue;">3.4 High Frequency Low Balance Accounts.</h3>**

In [27]:
frequency_balance = (
    df.groupby("AccountID")
      .agg(
          Total_Transactions=("TransactionID","count"),
          Average_Balance=("AccountBalance","mean")
      )
      .reset_index()
)

high_frequency = frequency_balance["Total_Transactions"].quantile(0.75)

low_balance = frequency_balance["Average_Balance"].quantile(0.25)

high_freq_low_balance = frequency_balance[
    (frequency_balance["Total_Transactions"]>=high_frequency) &
    (frequency_balance["Average_Balance"]<=low_balance)
]

high_freq_low_balance

,AccountID,Total_Transactions,Average_Balance
9,ACC15228,5,62944.415270
11,ACC15671,6,51776.654462
55,ACC31539,5,56982.964056
72,ACC39500,6,57445.826828
85,ACC45907,10,53323.047352
101,ACC51009,5,56690.771862
107,ACC53466,7,57309.855371
113,ACC57597,5,39875.536866
114,ACC57700,6,53388.047296
117,ACC58667,5,55285.627332


**<h3 style="color:blue;">3.5 Accounts with Negative or Near Zero Balance.</h3>**

In [29]:
balance_profile = (
    df.groupby("AccountID")["AccountBalance"]
      .mean()
      .reset_index(name="Average_Balance")
)

threshold = balance_profile["Average_Balance"].quantile(0.05)

negative_accounts = balance_profile[
    balance_profile["Average_Balance"]<=threshold
]

negative_accounts

,AccountID,Average_Balance
42,ACC28292,37550.014430
59,ACC32890,45439.899276
84,ACC45521,46750.408865
97,ACC49422,38841.595410
109,ACC54589,42538.305527
111,ACC55729,44902.541790
113,ACC57597,39875.536866
119,ACC61827,46163.402105
150,ACC77638,47234.380597
192,ACC99409,32962.941265
